In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from pzest.config import (
    load_config,
    IMAGE_ARCSINH_SCALE,
    IMAGE_CHANNEL_MEAN,
    IMAGE_CHANNEL_STD,
    CI_LOWER_PERCENTILE,
    CI_UPPER_PERCENTILE,
)
from pzest.evaluation.inference import predict_from_arrays
from pzest.models.deepz import DeepZ
from pzest.utils import load_checkpoint

## DeepZ Inference

Runs DeepZ inference on a user-provided HDF5 file containing HSC galaxy images. Outputs a results HDF5 file containing full redshift PDFs and a CSV with point estimates and credible intervals.

### Input HDF5 format

The input file must contain:
- `images` - shape (N, 5, 64, 64), float32, HSC grizy bands
- `object_id` - shape (N,), object identifiers

Optionally:
- `magnitudes` - shape (N, 5), HSC grizy CModel magnitudes

If `magnitudes` is absent, the model runs in image-only mode.

### Preprocessing flag

Set `preprocess_images = True` if your images are raw HSC flux values.
Set `preprocess_images = False` if your images are already preprocessed using the same pipeline as the DeepZ training data.

The DeepZ preprocessing pipeline applies:
1. Per-band arcsinh stretch: `arcsinh(x / sigma_noise)` using the noise scale factors defined in `pzest/config.py` as `IMAGE_ARCSINH_SCALE`
2. Per-channel standardisation using `IMAGE_CHANNEL_MEAN` and `IMAGE_CHANNEL_STD` from `pzest/config.py`

These constants were derived from the full GalaxiesML-Spectra training set. See `notebooks/02_preprocess.ipynb` for the full preprocessing pipeline.
If your images came from our processed HDF5 file (`data/processed/processed.hdf5`) they are already preprocessed - set `preprocess_images = False`.

USER SETTINGS - edit these before running

In [ ]:
# Path to your input HDF5 file
INPUT_HDF5 = Path(".path/to/your/file.hdf5")

# Output paths
OUTPUT_HDF5 = Path("path/to/your/inference_results.hdf5")
OUTPUT_CSV = Path("path/to/your/inference_results.csv")

# Set True if images are raw HSC flux values, False if already preprocessed
PREPROCESS_IMAGES = True

# Model checkpoint and matching config
# Each checkpoint has a corresponding archived config in config/
# Make sure these match:
#   baseline_best.pt  ->  baseline_config.yaml
#   model1_best.pt    ->  model1_config.yaml
#   model2_best.pt    ->  model2_config.yaml
#   model3_best.pt    ->  model3_config.yaml  (recommended)
CHECKPOINT_NAME = "model3_best.pt"
CONFIG_PATH = "../config/model3_config.yaml"

In [ ]:
config = load_config(CONFIG_PATH)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Config: {CONFIG_PATH}")
print(f"Checkpoint: {CHECKPOINT_NAME}")
print(f"Input: {INPUT_HDF5}")
print(f"Output: {OUTPUT_HDF5}")

Load Model and Checkpoint

In [ ]:
model = DeepZ(config)
checkpoint = load_checkpoint(
    model=model,
    path=config.paths.checkpoints_dir / CHECKPOINT_NAME,
    device=device,
)
model = model.to(device)
print(f"Loaded {CHECKPOINT_NAME} - epoch {checkpoint['epoch']} "
      f"(val σ_NMAD: {checkpoint['val_snmad']:.4f}")

Load and optionally preprocess images

In [ ]:
print("Loading input data...")

with h5py.File(INPUT_HDF5, "r") as f:
    images = f["images"][:]
    magnitudes = f["magnitudes"][:] if "magnitudes" in f else None

    if "object_id" in f:
        object_ids = f["object_id"][:]
    else:
        print("No object_id found - using sequential integer IDs")
        object_ids = np.arange(len(images))

n = len(images)
print(f"Loaded {n:,} galaxies")
print(f"Magnitudes available: {magnitudes is not None}")

if PREPROCESS_IMAGES:
    print("Preprocessing images...")
    arcsinh_scale = np.array(IMAGE_ARCSINH_SCALE, dtype=np.float32)
    channel_mean = np.array(IMAGE_CHANNEL_MEAN, dtype=np.float32)
    channel_std = np.array(IMAGE_CHANNEL_STD, dtype=np.float32)

    images = images.astype(np.float32)
    for c in range(5):
        images[:, c] = np.arcsinh(images[:, c] / arcsinh_scale[c])
        images[:, c] = (images[:, c] - channel_mean[c]) / channel_std[c]
    print("Preprocessing complete")

Run Inference

In [ ]:
bin_edges = np.linspace(
    config.model.redshift_min,
    config.model.redshift_max,
    config.model.num_bins + 1
)

pdfs, z_pred = predict_from_arrays(
    model=model,
    images=images,
    bin_edges=bin_edges,
    device=device,
    magnitudes=magnitudes,
    batch_size=config.train.batch_size,
)

bin_centres = 0.5 * (bin_edges[:-1] + bin_edges[1:])
cdf = np.cumsum(pdfs, axis=1)
z_lower = bin_centres[np.argmax(cdf >= CI_LOWER_PERCENTILE, axis=1)]
z_upper = bin_centres[np.argmax(cdf >= CI_UPPER_PERCENTILE, axis=1)]

print(f"Inference complete. {len(z_pred):,} galaxies processed")
print(f"z_pred range: [{z_pred.min():.3f}, {z_pred.max():.3f}]")


Plots

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.hist(z_pred, bins=50, edgecolor="none")
ax1.set_xlabel("Predicted redshift")
ax1.set_ylabel("Count")
ax1.set_title(f"Redshift distribution (n={len(z_pred):,})")

ax2.plot(bin_centres, pdfs[0], lw=1.5)
ax2.axvline(z_pred[0], color="green", lw=1.5, linestyle=":", label=f"z_pred={z_pred[0]:.3f}")
ax2.set_xlabel("Redshift")
ax2.set_ylabel("P(z)")
ax2.set_title("Example PDF (galaxy 0)")
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()

Save results

In [ ]:
# Save HDF5 with full PDFs
OUTPUT_HDF5.parent.mkdir(parents=True, exist_ok=True)
print(f"Save to {OUTPUT_HDF5}...")

with h5py.File(OUTPUT_HDF5, "w") as f:
    f.create_dataset("object_id", data=object_ids)
    f.create_dataset("z_pred", data=z_pred, dtype=np.float32)
    f.create_dataset("z_lower", data=z_lower, dtype=np.float32)
    f.create_dataset("z_upper", data=z_upper, dtype=np.float32)
    f.create_dataset("pdfs", data=pdfs, dtype=np.float32)
    f.create_dataset("bin_centres", data=bin_centres.astype(np.float32))

print("HDF5 saved")

# Save CSV with scalar columns only
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
print(f"Saving CSV to {OUTPUT_CSV}...")

df = pd.DataFrame({
    "object_id": object_ids,
    "z_pred": z_pred.round(6),
    "z_lower": z_lower.round(6),
    "z_upper": z_upper.round(6),
})
df.to_csv(OUTPUT_CSV, index=False)
print(f"CSV saved. {len(df):,} rows.")